# Aula Completa: Estruturas de Dados em C

## Objetivos
Ao final, você deverá explicar o que são estruturas de dados e TADs, comparar custos de operações, escolher uma estrutura para um problema e implementar exemplos em C entendendo cada trecho.

## 1. Conceitos fundamentais
**Dado** é um valor que representa informação. **Estrutura de dados** é uma forma de organizar dados e operações para armazená-los, consultá-los, inseri-los e removê-los. **Algoritmo** é uma sequência finita de passos para resolver um problema. Estrutura e algoritmo se complementam: uma estrutura conveniente torna operações importantes mais simples ou rápidas.

**Elemento** é o valor armazenado. **Índice** identifica uma posição, começando em `0` nos vetores C. **Nó** é uma unidade de estrutura encadeada, normalmente com dados e ponteiros. **Ponteiro** armazena endereço de memória; `NULL` representa ausência de endereço válido. **TAD (tipo abstrato de dado)** descreve comportamento e operações sem fixar como serão implementados: uma pilha pode usar vetor ou lista, mas continua obedecendo LIFO.

**Invariante** é uma regra que deve permanecer verdadeira. Exemplo: numa fila, inserções vão ao fim e remoções saem pelo início. **Capacidade** é o espaço reservado; **quantidade** é o número de elementos presentes. Distingui-los evita acessar memória fora dos limites.

### Complexidade (Big O)
Big O descreve como o custo cresce com `n`, quantidade de dados, e não mede segundos diretamente. `O(1)` é constante; `O(log n)` cresce lentamente ao dividir o problema; `O(n)` percorre os dados; `O(n log n)` aparece em ordenações eficientes; `O(n^2)` costuma comparar pares/aninhamentos.

| Operação | Exemplo | Custo típico |
|---|---|---|
| acesso por índice | `vetor[i]` | `O(1)` |
| busca não ordenada | percorrer vetor/lista | `O(n)` |
| busca binária | vetor ordenado | `O(log n)` |
| inserir/remover no meio de vetor | deslocar elementos | `O(n)` |
| empilhar/desempilhar no topo | pilha | `O(1)` |

O custo depende da operação e da implementação; não há uma estrutura melhor para todos os problemas.

## 2. Vetores
Um vetor (array) mantém elementos do mesmo tipo em posições consecutivas. C permite acesso direto por índice, mas seu tamanho declarado é fixo. A definição do NIST destaca esse acesso por índice; inserções no meio normalmente exigem deslocamentos.

No exemplo seguinte, `quantidade` conta as posições usadas, enquanto a capacidade é cinco. Observe o limite do laço: ele termina antes de `quantidade`, pois os índices válidos começam em zero.

```c
#include <stdio.h>

int main(void) {
    int valores[5] = {10, 20, 30}; // Reserva cinco inteiros; os demais iniciam em zero.
    int quantidade = 3;            // Só três posições são consideradas preenchidas.

    if (quantidade < 5) {           // Antes de inserir, verificamos se há espaço.
        valores[quantidade] = 40;   // quantidade vale 3: grava na quarta posição.
        quantidade++;               // Agora há quatro elementos válidos.
    }

    for (int i = 0; i < quantidade; i++) { // Visita do primeiro ao último elemento usado.
        printf("%d ", valores[i]);
    }
    printf("\\n");
    return 0;
}
```

**Leitura guiada:** `int valores[5]` reserva cinco inteiros; índices válidos são de `0` a `4`. O inicializador fornece três valores e zera os demais. A verificação impede ultrapassar a capacidade. A atribuição usa o próximo índice livre. Para acesso por índice, o custo é `O(1)`; para buscar um valor sem ordenação, `O(n)`.

## 3. Memória dinâmica e ponteiros
Variáveis locais comuns têm duração automática. Estruturas que precisam crescer durante a execução podem reservar espaço dinamicamente. `malloc` reserva bytes não inicializados e retorna `NULL` em falha; `free` devolve ao alocador um bloco previamente obtido. Nunca use um ponteiro após liberar seu bloco, nem libere o mesmo bloco duas vezes.

O padrão `sizeof *ponteiro` calcula o tamanho do tipo apontado. O código abaixo cria um vetor de quatro inteiros e libera a memória ao final.

```c
#include <stdio.h>
#include <stdlib.h>

int main(void) {
    size_t quantidade = 4;
    int *valores = malloc(quantidade * sizeof *valores); // Reserva espaço na heap.
    if (valores == NULL) { // malloc pode falhar; não dereferencie NULL.
        fprintf(stderr, "Falha ao alocar memoria.\\n");
        return 1;
    }
    for (size_t i = 0; i < quantidade; i++) {
        valores[i] = (int)(i + 1) * 10; // A aritmética de ponteiros permite esta notação de vetor.
    }
    printf("Terceiro valor: %d\\n", valores[2]);
    free(valores); // Libera exatamente uma vez a alocação associada.
    valores = NULL; // Marca que não há bloco válido acessível por esta variável.
    return 0;
}
```

**Por que testar e liberar?** Sem testar, uma alocação que falhou pode causar acesso inválido. Sem `free`, o bloco fica perdido enquanto o programa continua: é um vazamento de memória. A memória de `malloc` não é inicializada; atribua valores antes de lê-los. `calloc` aloca e zera; `realloc` pode redimensionar e mover um bloco, então seu uso exige preservar o ponteiro original até confirmar sucesso.

## 4. Lista simplesmente encadeada
Uma **lista encadeada** é uma sequência de nós em que cada nó guarda um elemento e um link (ponteiro) para o próximo. O NIST caracteriza a lista simplesmente encadeada dessa forma. O primeiro endereço fica no ponteiro `inicio`; o último nó aponta para `NULL`.

```text
inicio -> [10 | prox] -> [20 | prox] -> [30 | NULL]
```

Ela não exige posições contíguas nem capacidade fixa. Inserir no início custa `O(1)`, mas buscar, remover por valor ou chegar ao fim custa `O(n)`. Um vetor oferece acesso direto `O(1)` por índice, que a lista não oferece. Lista duplamente encadeada guarda também ponteiro para o anterior; lista circular liga o último nó de volta ao primeiro.

O exemplo abaixo implementa inserir no início, buscar, remover a primeira ocorrência, imprimir e liberar todos os nós. Repare no uso de `anterior` para religar a lista durante a remoção. Antes de liberar um nó, `liberar` guarda o endereço do próximo nó, pois não se pode acessar um objeto depois de `free`.

```c
#include <stdio.h>
#include <stdlib.h>

typedef struct No {
    int dado;              // O valor guardado neste nó.
    struct No *proximo;    // Endereço do próximo nó, ou NULL no último.
} No;

typedef struct {
    No *inicio;            // Ponto de entrada para percorrer toda a lista.
} Lista;

void inicializar(Lista *lista) {
    lista->inicio = NULL;  // Lista vazia não tem primeiro nó.
}

int inserir_inicio(Lista *lista, int valor) {
    No *novo = malloc(sizeof *novo); // Reserva dinamicamente um nó.
    if (novo == NULL) return 0;      // Informa falha de alocação.
    novo->dado = valor;              // Inicializa o campo de dados.
    novo->proximo = lista->inicio;   // Novo aponta para o antigo primeiro.
    lista->inicio = novo;            // O descritor agora aponta para o novo primeiro.
    return 1;
}

No *buscar(const Lista *lista, int valor) {
    No *atual = lista->inicio;       // Começa pelo primeiro nó.
    while (atual != NULL) {          // NULL marca o fim da cadeia.
        if (atual->dado == valor) return atual; // Retorna endereço se encontrou.
        atual = atual->proximo;      // Avança para o nó seguinte.
    }
    return NULL;                     // Não encontrou o valor.
}

int remover(Lista *lista, int valor) {
    No *atual = lista->inicio;       // Nó que estamos examinando.
    No *anterior = NULL;             // NULL indica que atual ainda é o primeiro.
    while (atual != NULL && atual->dado != valor) {
        anterior = atual;            // Salva o nó que ficará antes do removido.
        atual = atual->proximo;       // Continua a busca.
    }
    if (atual == NULL) return 0;      // Valor ausente: nada foi removido.
    if (anterior == NULL) {
        lista->inicio = atual->proximo; // Remove o primeiro nó.
    } else {
        anterior->proximo = atual->proximo; // Salta sobre o nó removido.
    }
    free(atual);                     // Devolve o nó removido ao alocador.
    return 1;
}

void exibir(const Lista *lista) {
    const No *atual = lista->inicio; // Não modificamos a lista ao percorrê-la.
    while (atual != NULL) {
        printf("%d -> ", atual->dado); // Exibe o dado corrente.
        atual = atual->proximo;       // Move-se ao próximo nó.
    }
    puts("NULL");                    // Mostra visualmente o fim.
}

void liberar(Lista *lista) {
    No *atual = lista->inicio;
    while (atual != NULL) {
        No *proximo = atual->proximo; // Salva antes do free.
        free(atual);                  // Libera o nó corrente.
        atual = proximo;              // Segue usando o endereço salvo.
    }
    lista->inicio = NULL;             // Restaura o invariante de lista vazia.
}

int main(void) {
    Lista numeros;
    inicializar(&numeros);            // Passamos endereço para alterar a lista original.
    inserir_inicio(&numeros, 30);
    inserir_inicio(&numeros, 20);
    inserir_inicio(&numeros, 10);     // Sequência resultante: 10 -> 20 -> 30.
    exibir(&numeros);
    printf("Busca 20: %s\\n", buscar(&numeros, 20) ? "encontrado" : "ausente");
    remover(&numeros, 20);            // Remove somente a primeira ocorrência.
    exibir(&numeros);
    liberar(&numeros);                // Não deixar memória dinâmica ocupada.
    return 0;
}
```

`lista->inicio` equivale a `(*lista).inicio`. `&numeros` fornece o endereço da variável local para as funções. A função `buscar` devolve `No *`: isso permite testar se o retorno é `NULL` e também acessar o nó enquanto ele permanecer válido.

## 5. Pilha (LIFO)
Uma **pilha** é uma coleção em que só o elemento mais recentemente inserido pode ser removido: **LIFO**, *Last In, First Out*. O NIST lista `push` e `pop` como operações básicas. Imagine pratos empilhados: acrescentamos e retiramos pelo topo.

Operações usuais: `push`/empilhar insere; `pop`/desempilhar remove e devolve o topo; `top`/`peek` consulta sem remover; `isEmpty` testa se está vazia. **Overflow** ocorre quando uma pilha limitada está cheia; **underflow**, ao tentar remover de uma pilha vazia. Aqui cada operação custa `O(1)`.

```c
#include <stdio.h>
#define CAPACIDADE 5

typedef struct {
    int dados[CAPACIDADE]; // Armazenamento limitado da pilha.
    int topo;              // Índice do último item; -1 significa vazia.
} Pilha;

void inicializar(Pilha *p) {
    p->topo = -1;          // Estabelece o invariante de pilha vazia.
}

int empilhar(Pilha *p, int valor) {
    if (p->topo == CAPACIDADE - 1) return 0; // Não sobrescrever o vetor.
    p->topo++;                              // Reserva a próxima posição.
    p->dados[p->topo] = valor;              // Coloca o valor no novo topo.
    return 1;                               // Operação concluída.
}

int desempilhar(Pilha *p, int *valor) {
    if (p->topo == -1) return 0;            // Não existe valor para remover.
    *valor = p->dados[p->topo];             // Copia o topo para a saída.
    p->topo--;                              // O elemento deixa de pertencer à pilha.
    return 1;
}

int main(void) {
    Pilha p;
    int removido;
    inicializar(&p);
    empilhar(&p, 10);                       // Primeiro a entrar.
    empilhar(&p, 20);
    empilhar(&p, 30);                       // Último a entrar: fica no topo.
    if (desempilhar(&p, &removido)) {
        printf("Saiu primeiro: %d\\n", removido); // 30, regra LIFO.
    }
    return 0;
}
```

O parâmetro `int *valor` é um parâmetro de saída: a função informa sucesso pelo retorno (`1` ou `0`) e grava o resultado no endereço recebido. Uma pilha pode ser implementada com vetor ou lista encadeada. Aplicações: desfazer/refazer, chamadas de funções, avaliação de expressões e histórico de navegação.

## 6. Fila (FIFO) e fila circular
Uma **fila** deixa acessar primeiro o item mais antigo: **FIFO**, *First In, First Out*. Segundo o NIST, adicionamos (enqueue) pela cauda e removemos (dequeue) pela cabeça. Exemplos: atendimento, impressão e tarefas em ordem de chegada.

Com vetor simples, remover do início deslocando os demais custa `O(n)`. Uma fila circular evita deslocamentos: índices avançam com resto `% capacidade`, reaproveitando o espaço. O campo `quantidade` distingue cheia (`quantidade == capacidade`) de vazia (`quantidade == 0`). Enfileirar e desenfileirar custam `O(1)`.

```c
#include <stdio.h>
#define CAPACIDADE 5

typedef struct {
    int dados[CAPACIDADE]; // Espaço físico do buffer circular.
    int inicio;            // Índice do próximo item a ser atendido.
    int quantidade;        // Número de itens atualmente enfileirados.
} Fila;

void inicializar(Fila *f) {
    f->inicio = 0;
    f->quantidade = 0;     // Ambos os valores definem uma fila vazia.
}

int enfileirar(Fila *f, int valor) {
    if (f->quantidade == CAPACIDADE) return 0; // Não há espaço livre.
    int fim = (f->inicio + f->quantidade) % CAPACIDADE; // Próxima posição lógica.
    f->dados[fim] = valor;                       // Adiciona ao final.
    f->quantidade++;
    return 1;
}

int desenfileirar(Fila *f, int *valor) {
    if (f->quantidade == 0) return 0;       // Underflow: fila vazia.
    *valor = f->dados[f->inicio];           // O mais antigo sai primeiro.
    f->inicio = (f->inicio + 1) % CAPACIDADE; // Avança e volta ao início se necessário.
    f->quantidade--;
    return 1;
}

int main(void) {
    Fila f;
    int atendido;
    inicializar(&f);
    enfileirar(&f, 100);
    enfileirar(&f, 200);
    enfileirar(&f, 300);
    if (desenfileirar(&f, &atendido)) {
        printf("Atendido: %d\\n", atendido); // 100: o primeiro que entrou.
    }
    enfileirar(&f, 400); // A posição liberada pode ser reutilizada circularmente.
    return 0;
}
```

O `% CAPACIDADE` faz o índice retornar a `0` após a última posição. A fila guarda sua própria quantidade, então não precisa reservar uma posição vazia para distinguir cheia de vazia. Em problemas reais, trate o retorno `0` de inserção e remoção.


## 7. Árvores e árvore binária de busca
Uma **árvore** representa relações hierárquicas. **Raiz** é o nó inicial, **filho** é um nó ligado a outro, **pai** é o nó que o referencia, e **folha** não possui filhos. Em uma árvore binária cada nó tem no máximo dois filhos: esquerdo e direito.

Uma **árvore binária de busca (ABB)** mantém valores menores à esquerda e maiores à direita (conforme a convenção adotada para duplicatas). O NIST define essa propriedade por subárvores e chaves. Busca, inserção e remoção podem custar `O(log n)` quando equilibrada, mas `O(n)` quando fica inclinada como uma lista. AVL e Red-Black são exemplos de árvores balanceadas.

O exemplo insere valores e percorre a árvore em ordem. A recursão precisa de um caso-base: quando a raiz da subárvore é `NULL`, a chamada retorna. O percurso esquerda-raiz-direita produz ordem crescente em uma ABB.

```c
#include <stdio.h>
#include <stdlib.h>

typedef struct NoArvore {
    int valor;                       // Chave armazenada.
    struct NoArvore *esquerda;       // Raiz da subárvore com valores menores.
    struct NoArvore *direita;        // Raiz da subárvore com valores maiores.
} NoArvore;

NoArvore *inserir(NoArvore *raiz, int valor) {
    if (raiz == NULL) {              // Encontrou a posição vazia para o novo nó.
        NoArvore *novo = malloc(sizeof *novo);
        if (novo == NULL) return raiz; // Sem memória: não altera esta subárvore.
        novo->valor = valor;
        novo->esquerda = NULL;       // Nó recém-criado não tem filhos.
        novo->direita = NULL;
        return novo;                 // Devolve raiz atualizada ao chamador.
    }
    if (valor < raiz->valor) {
        raiz->esquerda = inserir(raiz->esquerda, valor); // Desce à esquerda.
    } else if (valor > raiz->valor) {
        raiz->direita = inserir(raiz->direita, valor);   // Desce à direita.
    }                                    // Igual: este exemplo ignora duplicatas.
    return raiz;                         // Raiz da subárvore permanece a mesma.
}

void em_ordem(const NoArvore *raiz) {
    if (raiz == NULL) return;            // Caso-base: subárvore vazia.
    em_ordem(raiz->esquerda);            // Visita valores menores.
    printf("%d ", raiz->valor);          // Visita o nó corrente.
    em_ordem(raiz->direita);             // Visita valores maiores.
}

void liberar_arvore(NoArvore *raiz) {
    if (raiz == NULL) return;
    liberar_arvore(raiz->esquerda);      // Primeiro libera descendentes esquerdos.
    liberar_arvore(raiz->direita);       // Depois descendentes direitos.
    free(raiz);                          // Por último libera o nó corrente.
}

int main(void) {
    NoArvore *raiz = NULL;
    raiz = inserir(raiz, 8);
    raiz = inserir(raiz, 3);
    raiz = inserir(raiz, 10);
    raiz = inserir(raiz, 1);
    raiz = inserir(raiz, 6);
    printf("Ordem crescente: ");
    em_ordem(raiz);                      // Saída: 1 3 6 8 10
    printf("\\n");
    liberar_arvore(raiz);                // Cada nó alocado é liberado.
    return 0;
}
```

**Outros percursos:** pré-ordem visita raiz, esquerda e direita; pós-ordem visita esquerda, direita e raiz. Percursos são úteis para cópia, impressão hierárquica ou liberação. O programa ignora valores repetidos; uma aplicação pode, em vez disso, contar ocorrências ou colocar iguais sempre de um lado.

## 8. Busca, ordenação e tabela hash
**Busca linear** compara um valor com cada elemento até encontrar ou chegar ao fim: funciona com dados em qualquer ordem e custa `O(n)`. **Busca binária** compara com o meio e descarta metade do intervalo; requer vetor ordenado e custa `O(log n)`.

```c
int busca_binaria(const int vetor[], int tamanho, int alvo) {
    int inicio = 0;
    int fim = tamanho - 1;                  // Último índice válido.
    while (inicio <= fim) {                 // Ainda existe intervalo para examinar.
        int meio = inicio + (fim - inicio) / 2; // Evita somar diretamente os extremos.
        if (vetor[meio] == alvo) return meio;   // Retorna o índice da ocorrência.
        if (vetor[meio] < alvo) {
            inicio = meio + 1;               // Alvo só pode estar à direita.
        } else {
            fim = meio - 1;                  // Alvo só pode estar à esquerda.
        }
    }
    return -1;                               // Convenção para não encontrado.
}
```

`const int vetor[]` comunica que a função não altera o vetor. Quando `inicio > fim`, não restam posições possíveis. Para tamanho zero, `fim` vale `-1`, então o laço não executa e a função retorna `-1`.

**Ordenação** organiza os elementos por uma regra, como ordem crescente. Bubble sort é simples de acompanhar, mas custa `O(n^2)` no pior caso. Merge sort custa `O(n log n)` e precisa de memória auxiliar; quicksort tem custo médio `O(n log n)` e pior caso `O(n^2)` em implementações sem proteções. A busca binária só é válida se a ordenação e comparação usam a mesma regra.

Uma **tabela hash** transforma uma chave em posição por uma função hash. Inserção e busca têm custo médio esperado `O(1)`, mas colisões (chaves com o mesmo índice) exigem tratamento: encadeamento com listas ou endereçamento aberto. O pior caso pode ser `O(n)`. Hash não mantém automaticamente os elementos ordenados.

## 9. Comparação e escolha
| Estrutura | Acesso / busca | Inserção e remoção | Use quando |
|---|---|---|---|
| Vetor | índice `O(1)`; busca `O(n)` | meio `O(n)` | precisa acesso por posição e tamanho conhecido |
| Lista encadeada | busca `O(n)` | início `O(1)` | inserções no início e tamanho variável |
| Pilha | topo `O(1)` | topo `O(1)` | precisa da regra LIFO |
| Fila circular | frente `O(1)` | início/fim `O(1)` | processar na ordem FIFO |
| ABB | média `O(log n)` se equilibrada | média `O(log n)` se equilibrada | busca e percurso ordenado |
| Tabela hash | média `O(1)` | média `O(1)` | localizar rapidamente por chave |

Os custos são típicos e dependem do formato dos dados e da implementação. Escolha as operações mais frequentes primeiro: rápido para quê? Considere também ordem, consumo de memória, limites e simplicidade de manutenção.

## 10. Boas práticas e erros frequentes em C
1. Em um vetor de tamanho `n`, índices válidos são `0` até `n - 1`.
2. Inicialize ponteiros antes de usá-los; não dereferencie `NULL`.
3. Verifique o retorno de `malloc` antes de usar a memória.
4. Libere cada bloco dinâmico uma única vez e não acesse após `free`.
5. Guarde o próximo nó antes de liberar o nó atual durante uma travessia.
6. Atualize `inicio`/raiz quando operações mudarem o primeiro nó.
7. Trate coleção vazia, cheia, valor inexistente e falha de alocação.
8. Diferencie capacidade de quantidade e evite overflow ao calcular tamanhos.
9. Compile com avisos habilitados; avisos ajudam, mas não substituem testes.

```sh
gcc -std=c11 -Wall -Wextra -pedantic programa.c -o programa
./programa
```

## 11. Exercícios de fixação
1. No exemplo de vetor, implemente busca linear e retorne o índice da primeira ocorrência ou `-1`.
2. Adicione à lista uma operação para inserir no fim. Qual o custo para encontrar o último nó? Como um ponteiro `fim` altera esse custo?
3. Implemente uma função que conte os nós da lista e outra que remova todos os nós.
4. Use uma pilha para inverter uma palavra. Demonstre por que LIFO produz a inversão.
5. Mostre os elementos da fila circular sem removê-los, respeitando `inicio` e `quantidade`.
6. Insira `8, 3, 10, 1, 6` na ABB e desenhe-a. Escreva os percursos em pré-ordem, em ordem e pós-ordem.
7. Dê um exemplo de colisão em uma tabela hash e explique uma forma de resolvê-la.
8. Para cada operação dos exercícios, registre casos normal, vazio/cheio e não encontrado; estime a complexidade.

## Referências consultadas
- NIST, *Dictionary of Algorithms and Data Structures*: definições de [array](https://xlinux.nist.gov/dads/HTML/array.html), [linked list](https://xlinux.nist.gov/dads/HTML/linkedList.html), [stack](https://xlinux.nist.gov/dads/HTML/stack.html), [queue](https://xlinux.nist.gov/dads/HTML/queue.html) e [binary search tree](https://xlinux.nist.gov/dads/HTML/binarySearchTree.html). A consulta foi feita em 1 out. 2026.
- cppreference, [malloc (C)](https://en.cppreference.com/w/c/memory/malloc): alocação dinâmica, retorno em falha e liberação. A consulta foi feita em 1 out. 2026.
- Linux man-pages, [malloc(3)](https://man7.org/linux/man-pages/man3/malloc.3.html): comportamento de `malloc`, `calloc`, `realloc` e `free`, incluindo requisitos de propriedade e liberação.

## Fechamento
Estruturas de dados são modelos para organizar informação e definir operações. Para resolver um problema, identifique o que precisa ser rápido, selecione uma estrutura compatível, implemente seus invariantes, proteja os limites e trate a memória com cuidado. Ponteiros e alocação dinâmica permitem flexibilidade em C, mas também tornam responsabilidade do programa evitar acessos inválidos, vazamentos e uso após liberação.